# BrainWorks: Quantifying the Infiltrative Margin in Paediatric Diffuse Glioma

This notebook provides the training-free, signal-level evaluation code for our MICCAI 2026 submission. It reproduces the core experiments from the paper:
1. **Data Validation:** Pair checking for FLAIR and segmentation masks.
2. **Experiment 1 (AUC):** Boundary detectability via Wavelet vs. generic Sobel.
3. **Experiment 2 (Stratification):** Margin conspicuity analysis.
4. **Experiment 3 (Transition Profile):** Signal profile and infiltration ramp.
5. **Experiment 4 (Width Comparison):** Physical transition width (Gradient vs. Wavelet).
6. **Qualitative Visualization:** 2D edge detection mapping.
7. **Final Correlation & Paper Figures:** Spearman analysis and publication-ready plots.

In [3]:
from google.colab import drive
import os
import glob

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define data directory
DATA_DIR = '/content/drive/MyDrive/BraTS_Experimento'

# 3. Check pairings
patients = sorted([d for d in glob.glob(os.path.join(DATA_DIR, "*")) if os.path.isdir(d)])
print(f"Searching in: {DATA_DIR}\nPatients found: {len(patients)}\n")

errors = 0
for p in patients:
    patient_id = os.path.basename(p)
    flair = glob.glob(os.path.join(p, "*t2f.nii.gz"))
    seg = glob.glob(os.path.join(p, "*seg.nii.gz"))

    if not flair or not seg:
        print(f"❌ ERROR in {patient_id}: Missing files (FLAIR: {len(flair)}, SEG: {len(seg)})")
        errors += 1

if errors == 0:
    print("✅ DATA INTEGRITY VERIFIED: All patients have successfully paired FLAIR and segmentation masks. Ready to proceed.")
else:
    print(f"⚠️ WARNING: Found {errors} patients with missing files. Please review before proceeding.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Searching in: /content/drive/MyDrive/BraTS_Experimento
Patients found: 37

✅ DATA INTEGRITY VERIFIED: All patients have successfully paired FLAIR and segmentation masks. Ready to proceed.


## Experiment 1: Boundary Detectability (AUC)
Evaluating the local high-pass response. We compare the ROC AUC of a wavelet detail-energy detector against a generic Sobel gradient within the tumour neighbourhood.

In [4]:
import mechanistic_controls as mc
import nibabel as nib
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
rows_exp1 = []

print(f"Running Experiment 1 (AUC Detectability) for {len(patients)} patients. Please wait...")

for p in patients:
    patient_id = os.path.basename(p)
    f = glob.glob(os.path.join(p, "*t2f.nii.gz"))[0]
    s = glob.glob(os.path.join(p, "*seg.nii.gz"))[0]

    flair = nib.load(f).get_fdata().astype(np.float32)
    seg = nib.load(s).get_fdata().astype(np.int32)

    res = mc.analyze_case(flair, seg, rng)
    if res:
        res['patient_id'] = patient_id
        rows_exp1.append(res)

df_exp1 = pd.DataFrame(rows_exp1)
df_exp1.to_csv('/content/Experiment1_Results.csv', index=False)
print("\n✅ File 'Experiment1_Results.csv' successfully saved.")

print(f"\n{'Metric':<22}{'Mean ± Std':<20}")
print("-" * 42)
for col in df_exp1.select_dtypes(include=[np.number]).columns:
    print(f"{col:<22}{df_exp1[col].mean():.3f} ± {df_exp1[col].std():.3f}")

Running Experiment 1 (AUC Detectability) for 37 patients. Please wait...

✅ File 'Experiment1_Results.csv' successfully saved.

Metric                Mean ± Std          
------------------------------------------
AUC_wavelet_neigh     0.592 ± 0.095
AUC_sobel_neigh       0.635 ± 0.119
AUC_wavelet_brain     0.783 ± 0.070
AUC_sobel_brain       0.792 ± 0.081
AUC_wavelet_null      0.504 ± 0.048
R_wavelet             1.605 ± 0.726
AUC_aad               0.586 ± 0.081
AUC_ada               0.568 ± 0.056
AUC_add               0.534 ± 0.046
AUC_daa               0.561 ± 0.063
AUC_dad               0.527 ± 0.054
AUC_dda               0.528 ± 0.045
AUC_ddd               0.520 ± 0.040


## Experiment 2: Margin Stratification (Conspicuity)
Extracting the margin conspicuity index ($\sigma$) to demonstrate that detectability collapses as the margin becomes more diffuse.

In [5]:
import margin_stratification as ms

rows_exp2 = []
print(f"Running Experiment 2 (Conspicuity Stratification) for {len(patients)} patients...\n")

for p in patients:
    patient_id = os.path.basename(p)
    f = glob.glob(os.path.join(p, "*t2f.nii.gz"))[0]
    s = glob.glob(os.path.join(p, "*seg.nii.gz"))[0]

    flair = nib.load(f).get_fdata().astype(np.float32)
    seg = nib.load(s).get_fdata().astype(np.int32)

    res = ms.analyze(flair, seg)
    if res:
        res['patient_id'] = patient_id
        rows_exp2.append(res)

print("\n--- EXPERIMENT 2 CONCLUSIONS (Real Cohort Results) ---")
ms.cohort_report(rows_exp2)

pd.DataFrame(rows_exp2).to_csv('/content/Experiment2_Results.csv', index=False)
print("\n✅ File 'Experiment2_Results.csv' successfully saved.")

Running Experiment 2 (Conspicuity Stratification) for 37 patients...


--- EXPERIMENT 2 CONCLUSIONS (Real Cohort Results) ---
  AUC_wavelet: Spearman rho(sigma, AUC) = +0.765  (p=3.57e-08, n=37)
  AUC_sobel: Spearman rho(sigma, AUC) = +0.768  (p=2.91e-08, n=37)
  SHARP (high sigma)     AUC_wavelet=0.654  AUC_sobel=0.715  (n=19)
  DIFFUSE (low sigma)    AUC_wavelet=0.527  AUC_sobel=0.551  (n=18)

✅ File 'Experiment2_Results.csv' successfully saved.


## Experiment 3: Signed-Distance Transition Profiling
Generating millimetric signal profiles to quantify the infiltrative ramp of the paediatric margin.

In [6]:
import distance_profile as dp

pairs = []
for p in patients:
    patient_id = os.path.basename(p)
    flair = glob.glob(os.path.join(p, "*t2f.nii.gz"))[0]
    seg = glob.glob(os.path.join(p, "*seg.nii.gz"))[0]
    pairs.append((patient_id, flair, seg))

print(f"Calculating millimetric transition profiles for {len(pairs)} patients... (this may take a few minutes)")
rows_exp3, profiles = dp.run_cohort_nifti(pairs)

fig_path_real = '/content/distance_profile_fig_REAL.png'
dp.make_figure(rows_exp3, profiles, out_path=fig_path_real)

print(f"✅ SUCCESS! Figure generated and saved at: {fig_path_real}")

Calculating millimetric transition profiles for 37 patients... (this may take a few minutes)
figure -> /content/distance_profile_fig_REAL.png
✅ SUCCESS! Figure generated and saved at: /content/distance_profile_fig_REAL.png


## Experiment 4: Transition Width Comparison (Wavelet vs. Gradient)
Contrasting the physical transition width (mm) recovered by a contrast-normalised gradient against the wavelet detail-energy profile.

In [8]:
import wavelet_vs_gradient_width as wg

rows_exp4 = []
print(f"Running Experiment 4 (Wavelet vs Gradient Width) for {len(patients)} patients...\n")
print(f"{'Patient':<20} | {'Gradient (mm)':<15} | {'Wavelet (mm)':<15}")
print("-" * 55)

for p in patients:
    patient_id = os.path.basename(p)
    f = glob.glob(os.path.join(p, "*t2f.nii.gz"))[0]
    s = glob.glob(os.path.join(p, "*seg.nii.gz"))[0]

    fimg = nib.load(f)
    flair = fimg.get_fdata().astype(np.float32)
    seg = nib.load(s).get_fdata().astype(np.int32)
    spacing = tuple(float(x) for x in fimg.header.get_zooms()[:3])

    w_grad = wg.width_gradient(flair, seg, spacing)
    w_wav = wg.width_wavelet(flair, seg, spacing)

    print(f"{patient_id:<20} | {w_grad:<15.2f} | {w_wav:<15.2f}")

    rows_exp4.append({
        'patient_id': patient_id,
        'width_gradient_mm': w_grad,
        'width_wavelet_mm': w_wav
    })

pd.DataFrame(rows_exp4).to_csv('/content/Experiment4_Results.csv', index=False)
print("\n✅ File 'Experiment4_Results.csv' successfully saved. Main experiments completed!")

Running Experiment 4 (Wavelet vs Gradient Width) for 37 patients...

Patient              | Gradient (mm)   | Wavelet (mm)   
-------------------------------------------------------
BraTS-PED-00013-100  | 2.72            | 13.00          
BraTS-PED-00013-101  | 2.82            | 8.00           
BraTS-PED-00025-100  | 5.31            | 11.00          
BraTS-PED-00025-101  | 5.87            | 12.00          
BraTS-PED-00025-102  | 5.82            | 13.00          
BraTS-PED-00025-103  | 9.88            | 4.00           
BraTS-PED-00107-100  | 9.58            | 10.00          
BraTS-PED-00107-101  | 1.39            | 12.00          
BraTS-PED-00114-100  | 3.18            | 14.00          
BraTS-PED-00114-101  | 8.18            | 23.00          
BraTS-PED-00115-100  | 6.59            | 12.00          
BraTS-PED-00115-101  | 3.90            | 12.00          
BraTS-PED-00117-100  | 9.08            | 10.00          
BraTS-PED-00117-101  | 2.21            | 5.00           
BraTS-PED-00118-100 

## Qualitative Visualization
Extracting 2D slices to visually compare the radiologist's ground truth against Sobel and Wavelet edge detection responses.

In [9]:
import matplotlib.pyplot as plt
import pywt
from scipy.ndimage import sobel
import scipy.ndimage

def get_largest_tumor_slice(seg):
    z_counts = (seg > 0).sum(axis=(0, 1))
    return np.argmax(z_counts)

def compute_sobel_2d(slice_img):
    gx = sobel(slice_img, axis=0)
    gy = sobel(slice_img, axis=1)
    return np.sqrt(gx**2 + gy**2)

def compute_wavelet_2d(slice_img):
    coeffs = pywt.dwt2(slice_img, 'db1')
    cA, (cH, cV, cD) = coeffs
    return np.sqrt(cH**2 + cV**2 + cD**2)

example_patients = patients[:2]

fig, axes = plt.subplots(len(example_patients), 4, figsize=(16, 4 * len(example_patients)))
if len(example_patients) == 1:
    axes = [axes]

print("Generating qualitative visualization collage for the manuscript...\n")

for i, p in enumerate(example_patients):
    patient_id = os.path.basename(p)
    f = glob.glob(os.path.join(p, "*t2f.nii.gz"))[0]
    s = glob.glob(os.path.join(p, "*seg.nii.gz"))[0]

    flair = nib.load(f).get_fdata().astype(np.float32)
    seg = nib.load(s).get_fdata().astype(np.int32)

    z = get_largest_tumor_slice(seg)
    flair_slice = flair[:, :, z]
    seg_slice = seg[:, :, z] > 0

    sobel_slice = compute_sobel_2d(flair_slice)
    wavelet_slice = compute_wavelet_2d(flair_slice)

    axes[i][0].imshow(np.rot90(flair_slice), cmap='gray')
    axes[i][0].set_title(f"{patient_id}\nOriginal FLAIR")
    axes[i][0].axis('off')

    axes[i][1].imshow(np.rot90(flair_slice), cmap='gray')
    axes[i][1].contour(np.rot90(seg_slice), colors='red', linewidths=1)
    axes[i][1].set_title("Ground Truth (Radiologist)")
    axes[i][1].axis('off')

    axes[i][2].imshow(np.rot90(sobel_slice), cmap='magma')
    axes[i][2].set_title("Sobel Edge Detection")
    axes[i][2].axis('off')

    wavelet_resized = scipy.ndimage.zoom(wavelet_slice, 2, order=0)
    h, w = flair_slice.shape
    wavelet_resized = wavelet_resized[:h, :w]

    axes[i][3].imshow(np.rot90(wavelet_resized), cmap='magma')
    axes[i][3].set_title("Wavelet Edge Detection")
    axes[i][3].axis('off')

plt.tight_layout()
plt.savefig('/content/Qualitative_Comparison_Exp1.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ SUCCESS! Image saved as 'Qualitative_Comparison_Exp1.png'. Ready for LaTeX inclusion.")

Generating qualitative visualization collage for the manuscript...


✅ SUCCESS! Image saved as 'Qualitative_Comparison_Exp1.png'. Ready for LaTeX inclusion.


## Final Integration: Conspicuity ($\sigma$) vs Transition Width
Merging the results from Experiment 2 and 4 to test for statistical independence between the strength of the edge and its spatial extent.

In [10]:
from scipy.stats import spearmanr

df_exp2 = pd.read_csv('/content/Experiment2_Results.csv')
df_exp4 = pd.read_csv('/content/Experiment4_Results.csv')

df_exp4.columns = df_exp4.columns.str.strip().str.lower()
df_exp2.columns = df_exp2.columns.str.strip().str.lower()

col_id_exp4 = [c for c in df_exp4.columns if 'pacient' in c or 'patient' in c][0]
df_exp4 = df_exp4.rename(columns={col_id_exp4: 'patient_id'})

df_fusion = pd.merge(df_exp2, df_exp4, on='patient_id')

col_grad = [c for c in df_fusion.columns if 'gradient' in c or 'mm' in c][0]

sigma = df_fusion['sigma']
width = df_fusion[col_grad]

rho, p_value = spearmanr(sigma, width)

print("--- DEFINITIVE CROSS-ANALYSIS: CONSPICUITY (\u03C3) vs WIDTH (mm) ---")
print(f"Spearman rho(\u03C3, width) = {rho:+.3f} (p={p_value:.3g})")

if rho <= -0.4 and p_value < 0.05:
    print("\n🏆 SCENARIO A: Significant correlation found. Clean causal chain established.")
else:
    print("\n⚖️ SCENARIO B: No strong correlation. Conspicuity and transition width are independent margin properties.")

--- DEFINITIVE CROSS-ANALYSIS: CONSPICUITY (σ) vs WIDTH (mm) ---
Spearman rho(σ, width) = -0.216 (p=0.198)

⚖️ SCENARIO B: No strong correlation. Conspicuity and transition width are independent margin properties.


## Final Paper Figure Generation
Compiling the definitive multi-panel vector graphic (`.pdf`) representing the transition profile distribution across the cohort.

In [11]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def _normalised_outer_profile(c, I, edge_band=1.5, plateau_band=2.0):
    ok = ~np.isnan(I)
    c, Iv = c[ok], I[ok]
    if len(c) < 5:
        return None, None
    edge_sel = (c >= -edge_band) & (c <= edge_band)
    I_edge = np.median(Iv[edge_sel]) if edge_sel.sum() >= 1 else Iv[np.argmin(np.abs(c))]
    I_out = np.median(Iv[c >= c.max() - plateau_band])
    denom = I_edge - I_out
    if abs(denom) < 1e-6:
        return None, None
    norm = (Iv - I_out) / denom
    return c, norm

def make_figure_paper_final(rows, profiles, out_path="distance_profile_fig.pdf"):
    widths = {r['patient_id']: r['transition_width_mm'] for r in rows
              if np.isfinite(r['transition_width_mm'])}
    if not widths:
        print("No valid widths found."); return

    ids_sorted = sorted(widths, key=widths.get)
    sharp, diff = ids_sorted[0], ids_sorted[-2]

    plt.rcParams.update({'font.size': 10, 'axes.spines.top': False,
                         'axes.spines.right': False})
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

    for pid, col, lab in ((sharp, '#1f77b4', f'sharpest ({widths[sharp]:.1f} mm)'),
                          (diff, '#d62728', f'diffuse example ({widths[diff]:.1f} mm)')):
        c, I, G = profiles[pid]
        cc, norm = _normalised_outer_profile(c, I)
        if cc is not None:
            # Custom crop filter to render from -4 to +12 mm only
            mask = (cc >= -4) & (cc <= 12)
            ax[0].plot(cc[mask], norm[mask], color=col, lw=2.2, label=lab)

    ax[0].axvline(0, color='k', ls='--', lw=1, label='radiologist contour')
    ax[0].axhspan(0.1, 0.9, color='gray', alpha=0.10)

    ax[0].set_xlim(-4, 12)
    ax[0].set_ylim(0, 1.05)

    ax[0].set_xlabel('signed distance to contour (mm)\n$\\leftarrow$ inside      outside $\\rightarrow$')
    ax[0].set_ylabel('normalised FLAIR intensity')
    ax[0].set_title('(a) Transition profile: sharp vs diffuse')
    ax[0].legend(fontsize=8, frameon=False)

    w = np.array(list(widths.values()))
    ax[1].hist(w, bins=8, color='#2ca02c', alpha=0.85, edgecolor='k', linewidth=0.5)
    ax[1].axvline(np.median(w), color='#d62728', lw=1.8,
                  label=f'median {np.median(w):.1f} mm')
    ax[1].set_xlabel('transition width (mm)')
    ax[1].set_ylabel('number of patients')
    ax[1].set_title('(b) Infiltrative width across cohort')
    ax[1].legend(fontsize=8, frameon=False)

    off = np.array([r['peak_offset_mm'] for r in rows
                    if np.isfinite(r['peak_offset_mm'])])
    ax[2].hist(off, bins=8, color='#7570b3', alpha=0.85, edgecolor='k', linewidth=0.5)
    ax[2].axvline(0, color='k', ls='--', lw=1, label='on the contour')
    ax[2].set_xlabel('gradient peak offset (mm)\n$\\leftarrow$ inside      outside $\\rightarrow$')
    ax[2].set_ylabel('number of patients')
    ax[2].set_title('(c) Location of the image edge')
    ax[2].legend(fontsize=8, frameon=False)

    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)
    print("✅ Completed! Final publication figure generated at:", out_path)

print("Generating cropped transition figure...")
make_figure_paper_final(rows_exp3, profiles, "/content/distance_profile_fig.pdf")

Generating cropped transition figure...
✅ Completed! Final publication figure generated at: /content/distance_profile_fig.pdf
